# RAG Retrieval Layer

Builds retrieval and grounded generation over the valuation method corpus.

**Why this exists.** The valuation pipeline formats the entire practice dataset into one
string and sends all of it on every question. That is correct for numbers — you want the
*exact* revenue figure, not the four rows most similar to the phrase "revenue." But it left
a gap: when the model explained *why* an expense qualified as an add-back, it had nothing to
draw on. Those rules lived in regex patterns and hardcoded constants, where the model could
not see them. Nothing in that arrangement could state why an add-back qualified, cite a
source, or be checked for consistency.

Writing the rules down as text makes them retrievable — and makes it possible to check
whether a generated justification actually rests on the rule it cites.

| Question | Path | Why |
|---|---|---|
| "What was 2023 revenue?" | existing deterministic lookup | exactness matters; retrieval would be a downgrade |
| "Why is automobile only 25%?" | retrieval (this notebook) | the answer is a passage, not a cell |

**Two corpora, not one.**

| File | Contains | Shared across clients |
|---|---|---|
| `method_standards.md` | the rules — reusable at any practice | yes |
| `roselle_engagement.md` | this client's figures and findings | **no** |

Professional-services confidentiality requires information barriers between matters, so
engagement material must never surface in another client's answer. That isolation is
enforced in §4 below.

Kept separate from the main pipeline because retrieval is independently testable — no API
key, no CSVs, no practice data. Only the two markdown files.

## 1. Chunking

Retrieval searches **chunks**, not documents. Each chunk becomes one vector by averaging its
words, so a chunk covering four topics produces a blurred average close to nothing in
particular. Too small fails the other way: `$9,114.00 in FY2022` is precise and meaningless
without the sentence defining it.

The target is a passage that answers one question completely.

| Approach | Use when |
|---|---|
| Fixed size (every N chars) | text has no structure; accepts mid-sentence cuts |
| Paragraph | structure exists but headings don't |
| **Section / heading** | the document was written with meaningful headings |
| Semantic (detect topic shift) | structure is unreliable and accuracy justifies the cost |

Section-based here, because both documents were drafted so each section answers one
question.

**Breadcrumbs.** A bare heading like `## 5. Known failure modes` has no body of its own —
only subsections. Chunked naively its children survive but lose their parent, producing a
chunk labeled only `FY2022` with nothing saying *owner compensation*.

So each chunk carries its parent heading, written **into the chunk text** rather than stored
beside it. Metadata in a separate field is not embedded and cannot affect whether a chunk is
found. This generalizes: anything that should influence retrieval must be inside the
embedded text.

**Splitting.** Sections over 2,000 characters split at paragraph boundaries — never
mid-table. Two thresholds, not one: 2,000 triggers a split, 1,400 sizes the results. If both
were 2,000, a 2,100-character section would yield one full segment and a useless
100-character runt.

Sizes are in characters. Embedding models cap on *tokens* and truncate silently past the
limit — worse than an error, since nothing reports it. At roughly 4 characters per token,
1,400 characters is about 350 tokens, well under the 512 ceiling.

In [ ]:
## Cell 1 — parse both corpora into chunks

import re, json
from pathlib import Path

SOURCES = {"method": "method_standards.md", "roselle": "roselle_engagement.md"}
MAX_SEGMENT, OVERSIZED = 1400, 2000


def pack(paragraphs, limit):
    """Group whole paragraphs into segments under `limit` chars. Never splits a paragraph."""
    segs, cur = [], []
    for p in paragraphs:
        cand = cur + [p]
        if cur and sum(len(x) for x in cand) + 2 * len(cur) > limit:   # +2 per "\n\n" rejoin
            segs.append(cur); cur = [p]
        else:
            cur = cand
    if cur:
        segs.append(cur)
    return segs


chunks = []
for corpus, path in SOURCES.items():
    # encoding="utf-8" is not optional. Without it Python falls back to the platform
    # default (cp1252 on Windows) and every em dash arrives as "â€”" — which then gets
    # embedded, silently degrading the vectors.
    text = Path(path).read_text(encoding="utf-8")
    h2 = None

    # Split BEFORE each level-2/3 heading. \n(?=...) is a lookahead: consume the newline,
    # leave the "##" attached to its section. #{2,3} excludes the "#" title line.
    for part in re.split(r'\n(?=#{2,3} )', text):
        part = part.strip()
        if not part.startswith('#'):
            continue

        line    = part.split('\n', 1)[0]
        level   = len(line) - len(line.lstrip('#'))
        heading = line.lstrip('# ').strip()
        if level == 2:
            h2, crumb = heading, heading
        else:
            crumb = f"{h2} > {heading}" if h2 else heading

        body = part.split('\n', 1)[1].strip() if '\n' in part else ''
        if not body:
            continue                                  # bare heading survives in children's crumbs

        pieces = [{"heading": crumb, "text": f"{crumb}\n\n{body}"}]
        if len(pieces[0]["text"]) > OVERSIZED:
            pieces = []
            for i, seg in enumerate(pack(body.split("\n\n"), MAX_SEGMENT)):
                lab = crumb if i == 0 else f"{crumb} (cont. {i+1})"
                pieces.append({"heading": lab, "text": f"{lab}\n\n" + "\n\n".join(seg)})

        for p in pieces:
            p["client"] = None if corpus == "method" else corpus   # None = shared
            p["source"] = path
            chunks.append(p)

# ids encode the corpus, so a citation shows at a glance which document it came from
counters = {}
for c in chunks:
    pre = "method" if c["client"] is None else c["client"]
    counters[pre] = counters.get(pre, -1) + 1
    c["id"] = f"{pre}-{counters[pre]:02d}"

for c in chunks:
    print(f"  {c['id']:<12} {len(c['text']):>5}  {c['heading'][:60]}")
print(f"\n{len(chunks)} chunks, max {max(len(c['text']) for c in chunks)} chars")

## 2. Verifying the separation

The whole confidentiality guarantee rests on method chunks containing no client material.
That is an assumption about how the source document was written, so test it rather than
trust it.

Searching for client fingerprints — names and distinctive figures — inside the shared corpus
is a cheap check that catches an authoring mistake before it becomes a data leak.

Saving to JSON freezes the chunk set. If chunking drifts between runs, retrieval quality
drifts with it and you cannot tell whether a change came from a new model or different
chunks.

In [ ]:
## Cell 2 — check invariants and persist

FINGERPRINT = re.compile(
    r'Seybold|Chang|Hassenplug|Roselle|9,114|200,115|199,992|195,700|20,881|133,851')

leaked = [c["id"] for c in chunks if c["client"] is None and FINGERPRINT.search(c["text"])]

assert not leaked,                                       f"client data in shared corpus: {leaked}"
assert all(len(c["text"]) <= OVERSIZED for c in chunks), "a chunk is still oversized"
assert len({c["id"] for c in chunks}) == len(chunks),    "duplicate chunk ids"
assert all(c["heading"] in c["text"] for c in chunks),   "breadcrumb missing from text"

Path("chunks.json").write_text(json.dumps(chunks, indent=2, ensure_ascii=False),
                               encoding="utf-8")

shared = sum(1 for c in chunks if c["client"] is None)
print(f"OK — {len(chunks)} chunks ({shared} shared, {len(chunks)-shared} client-specific)")
print(f"     {sum(len(c['text']) for c in chunks):,} characters -> chunks.json")

---

# Stage 2 — Embedding

Retrieval compares *meaning*, not words. A question about why payroll tax was excluded should
find a chunk discussing employer Social Security even though the two share almost no
vocabulary. That requires converting text into vectors positioned so nearby means similar.

**Model: `all-MiniLM-L6-v2`** — ~90 MB, Apache-2.0, CPU-fast, 384 dimensions. Larger models
score higher, but at this corpus size retrieval failures come from chunking, not from the
model. Running locally also means the corpus never leaves the machine, which matters when it
names a practice and its owners' compensation.

**Driving the model directly rather than calling `.encode()`.** `sentence-transformers`
offers a one-line shortcut that does everything below. It is the right choice in production
and teaches nothing, so the pipeline is written out: tokenize, forward pass, pool, normalize.

```
pip install torch transformers
```

Weights download on first use and cache in `~/.cache/huggingface`. Later runs are offline.

In [ ]:
## Cell 3 — load the tokenizer and model

import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
chunks = json.loads(Path("chunks.json").read_text(encoding="utf-8"))

# A tokenizer and its model are a matched pair. Each model was trained against one specific
# vocabulary; pairing a model with a different tokenizer produces plausible garbage, not an error.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)

# eval() disables dropout. Without it the same input yields slightly different vectors on
# each call — nothing fails, results are just quietly non-reproducible.
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"{MODEL_NAME}\ndevice {device} | hidden {model.config.hidden_size} "
      f"| max tokens {tokenizer.model_max_length} | chunks {len(chunks)}")

## 3. Tokenizing, pooling, normalizing

**Tokenizing.** Models consume integer ids, not text. Every sequence in a batch must be the
same length to form a rectangular tensor, so shorter ones get **padded**. The
`attention_mask` records which positions are real (1) and which are padding (0).

**Pooling — the part to get right.** The model returns one vector *per token*. Collapsing
them into one vector per chunk by averaging naively includes the padding positions. A chunk
with 81 real tokens in a 450-wide batch is 82% filler; averaging all 450 makes its identity
mostly `[PAD]`. Nothing errors — short chunks just drift toward a meaningless shared point.
The mask zeroes padding and divides by the count of *real* tokens only.

**`torch.no_grad()`** turns off gradient tracking, which exists for training and is pure
overhead during inference.

**Normalizing** scales each vector to unit length, which makes cosine similarity a plain dot
product — why search later is one matrix multiply.

In [ ]:
## Cell 4 — embed the corpus

def mean_pool(hidden_states, attention_mask):
    """Collapse per-token vectors into one per sequence, ignoring padding.

    hidden_states : (batch, tokens, hidden)
    attention_mask: (batch, tokens)  1 = real, 0 = padding
    returns       : (batch, hidden)
    """
    m = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
    return (hidden_states * m).sum(1) / m.sum(1).clamp(min=1e-9)


def embed(texts):
    """Tokenize -> forward pass -> masked mean pool -> L2 normalize."""
    enc = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc)
    pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
    return torch.nn.functional.normalize(pooled, p=2, dim=1)


embeddings = embed([c["text"] for c in chunks])
print(f"embeddings {tuple(embeddings.shape)}  (chunks x hidden)")
print(f"unit norms : {bool(torch.allclose(embeddings.norm(dim=1), torch.ones(len(chunks), device=device)))}")

---

# Stage 3 — Search

## 4. Dual index, and client isolation

**The problem.** Searching chunk bodies alone fails on topic-level questions. Querying
`"open items"` — the literal heading of a section, present verbatim in that chunk —
returned the *wrong* chunk at a score of 0.098.

**The cause is dilution.** Mean pooling averages every token equally. That chunk runs ~350
tokens, of which the heading is 3, so the heading contributes under 1% of the vector. The
other 99% is dollar figures and surnames, and the vector sits there instead. Embedding the
same heading alone scored **0.850** against the correct chunk — same model, same text, no
dilution.

This also rules out the obvious fix: appending a summary line would add ~1% more signal and
get averaged away the same way.

**So each chunk gets two vectors.**

| Index | Built from | Catches |
|---|---|---|
| body | full chunk text | specifics — a surname, a dollar amount |
| heading | breadcrumb only | topics — `"open items"`, `"partial years"` |

Scored against both, keeping whichever is higher. Genuinely complementary: a surname appears
in no heading and only the body finds it, while `"open items"` is drowned in the body and
only the heading finds it.

**Scores are not comparable across the two paths.** Heading matches score systematically
higher because short text matches short text. A fixed cutoff like "reject below 0.4" would
discard every body match while admitting weak heading ones. Trust the ranking, not the value.

**Client isolation.** Chunks belonging to another client are set to `-inf` before ranking, so
they cannot be returned at any position. Not "unlikely" — impossible. That is the difference
between a policy and a guarantee, and it is why the tagging is derived from which file a
chunk came from rather than guessed from its heading.

**Known limitation.** `"What is still unresolved?"` fails on both indexes — the model does
not place *unresolved* near *open items*. Abstract synonymy is where small embedding models
are weakest, and no index restructuring fixes it.

In [ ]:
## Cell 5 — dual-index search with client isolation

head_vecs = embed([c["heading"] for c in chunks])


def search(question, k=3, client="roselle"):
    q    = embed([question])
    body = (q @ embeddings.T)[0]        # full text: catches specifics
    head = (q @ head_vecs.T)[0]         # heading only: catches topics
    best = torch.maximum(body, head)    # judge each chunk on its better evidence

    # Other clients' chunks become unreachable at any ranking.
    allowed = [i for i, c in enumerate(chunks) if c["client"] in (None, client)]
    gate = torch.full_like(best, float("-inf"))
    gate[allowed] = 0.0
    best = best + gate

    top = best.argsort(descending=True)[:k]
    return [(float(best[i]), "head" if head[i] > body[i] else "body", chunks[i]) for i in top]


for q in ["open items", "Why is the automobile add-back only 25 percent?",
          "Why was payroll tax taken out of the owner's compensation figure?"]:
    print(f"\nQ: {q}")
    for s, via, c in search(q):
        print(f"   {s:.3f}  [{via}]  {c['id']:<12} {c['heading'][:50]}")

---

# Stage 4 — Generation

## 5. Answering from the retrieved chunks

Retrieval finds the right chunks. It does not answer the question. This is the second half of
RAG: hand those chunks to a language model and have it write an answer *from them*.

The model runs on someone else's servers, so unlike everything above this can fail for
reasons unrelated to your code — the service can be busy, the key can be wrong, the network
can drop. Hence three cells rather than one.

| Cell | Does | Run |
|---|---|---|
| 6A | stores the key, picks the model | once per session |
| 6B | sends 5 tokens to test the connection | whenever something breaks |
| 6C | the actual question-answering | freely |

Keeping the key in its own cell means re-running a question does not re-prompt for it.
Keeping the connection test separate means a network hiccup is distinguishable from a bug.

In [ ]:
## Cell 6A — configuration (run once per session)

import os, getpass, requests, time

# Read from the environment if set, otherwise prompt. Never written into the notebook,
# so the file can be shared without leaking a credential.
API_KEY = os.environ.get("OPENROUTER_API_KEY") or getpass.getpass("OpenRouter API key: ")

# Changing this one string swaps the model; the API is identical across all of them.
MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"
print("key loaded:", API_KEY[:8] + "...", "| model:", MODEL)

### 6B — connection test

Answers one question: is the problem the service, or my code?

| Code | Meaning | Do |
|---|---|---|
| 200 | working | continue |
| 401 | key rejected | check the key |
| 429 | too many requests | wait, or switch models |
| 5xx | their server broke | wait |

**About 429.** Free models are shared. When the pool is saturated everyone is refused —
including on your first request of the day. It is not a quota you used up. If the error body
reads `limit_source: upstream_provider_shared_pool`, the model is simply busy.

Models served by several hosting providers survive this better than models served by one,
which is why a lower-rated model sometimes works when a higher-rated one does not.

In [ ]:
## Cell 6B — connectivity probe across candidate models

CANDIDATES = [
    "nvidia/nemotron-3-ultra-550b-a55b:free",
    "z-ai/glm-5.2:free",
    "google/gemma-4-31b-it:free",
    "google/gemma-4-26b-a4b-it:free",
]

for m in CANDIDATES:
    try:
        r = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization": f"Bearer {API_KEY}"},
            json={"model": m, "max_tokens": 5,
                  "messages": [{"role": "user", "content": "Reply with: ok"}]},
            timeout=20)
        if r.status_code == 200:
            print(f"OK        {m}  (provider: {r.json().get('provider')})")
        else:
            why = r.json().get("error", {}).get("metadata", {}).get("limit_source", "")
            print(f"{r.status_code:<9} {m}  {why}")
    except Exception as e:
        print(f"ERROR     {m}  {type(e).__name__}")
    time.sleep(1)                    # stay under the per-minute limit while probing

### 6C — the RAG answer

Where retrieval and generation join: `search()` returns chunks, they are formatted into a
numbered SOURCES block, and that block plus the question plus a set of rules go to the model.

**The rules are the whole point.** Without them the model answers from general knowledge
about dentistry and accounting — fluently, confidently, and with no way to tell which parts
came from your document. Rule 5 exists because an abstention that describes what the sources
*do* contain can disclose client material while refusing.

**`temperature=0`** requests deterministic output. Note this is only *near*-deterministic on
hosted models: requests route to different providers and floating-point accumulation varies,
so identical inputs can still produce slightly different wording. Worth knowing before
building an eval set — a single before-and-after comparison cannot distinguish a real
improvement from drift.

**Retry on 429 only.** Rate limiting is temporary and worth waiting out. A rejected key is
not — retrying it five times wastes a minute before reporting the wrong problem.

In [ ]:
## Cell 6C — generation

SYSTEM = """You answer questions about a dental practice valuation using ONLY the numbered \
sources provided. Follow these rules exactly:

1. Every factual claim must come from a source. Cite it inline as [source-id].
2. If the sources do not contain the answer, say so plainly. Do not fill the gap from \
general knowledge about valuation or accounting.
3. Quote figures exactly as written. Do not round, recalculate, or infer new numbers.
4. Be concise.
5. If you cannot answer, reply only: "Not covered by the available sources." Do not \
describe what the sources do contain."""


def answer(question, k=3, client="roselle", retries=3):
    hits = search(question, k, client)
    context = "\n\n".join(
        f"[{c['id']}] {c['heading']}\n{c['text'].split(chr(10)*2, 1)[1]}"
        for _, _, c in hits)

    for attempt in range(retries):
        r = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization": f"Bearer {API_KEY}"},
            json={"model": MODEL, "temperature": 0,
                  "messages": [{"role": "system", "content": SYSTEM},
                               {"role": "user",
                                "content": f"SOURCES:\n\n{context}\n\nQUESTION: {question}"}]},
            timeout=120)

        if r.status_code == 200:
            return (r.json()["choices"][0]["message"]["content"],
                    [(round(s, 3), c["id"]) for s, _, c in hits])
        if r.status_code != 429:                      # only 429 is worth retrying
            raise RuntimeError(f"{r.status_code}: {r.text[:300]}")
        wait = 15 * (attempt + 1)
        print(f"rate-limited, waiting {wait}s")
        time.sleep(wait)

    raise RuntimeError("still rate-limited after retries")

---

# Stage 5 — Guardrails

## 6. Checking the answer against the sources

The model was *told* to cite everything and invent nothing. This checks whether it did.

A wrong answer looks exactly like a right one — the model writes fluently either way, so
"it sounded correct" is not verification.

Most guardrail approaches ask a second model to judge the first. That works, but it costs
another API call, adds latency, and the judge can hallucinate too. For financial documents
there is a cheaper first line: deterministic checks that are free, instant, and cannot
themselves be wrong.

| Check | Catches |
|---|---|
| `fabricated_citations` | cites an id it was never given |
| `unsupported_numbers` | any figure absent from the retrieved text |
| `leaked_in_refusal` | source content disclosed inside an abstention |
| `uncited` | answered with no citation and no refusal |

The number check matters most. A model that writes *"$8,432 of personal use"* when that
figure appears nowhere in the sources has invented it — and invented financial figures are
the failure that actually costs you.

Numbers are normalized so `$9,114.00`, `9114` and `9,114.0` compare equal. Citation markers
are stripped first, or `[method-06]` reads as the number 6 and every answer fails. Numbers
echoed from the question are excluded from the leak check — the user already had those.

**Two things this does not catch.** It cannot tell whether the *prose* is accurate: an answer
can cite the right chunk, use no numbers, and still describe the rule backwards. And it flags
legitimate arithmetic, since a computed result is by definition not in the sources — correct
behaviour given rule 3, but a false-positive source rather than a bug.

In [ ]:
## Cell 7 — deterministic guardrails

CITE    = re.compile(r'\[((?:method|roselle)-\d+)\]')
NUM     = re.compile(r'\$?\d[\d,]*(?:\.\d+)?%?')
ABSTAIN = re.compile(r"not covered|do(?: not|n't) (?:contain|have)"
                     r"|not (?:in|covered|available|present)|cannot answer|no information", re.I)


def _numbers(text):
    """Extract numbers normalized so $9,114.00 / 9114 / 9,114.0 compare equal."""
    out = set()
    for tok in NUM.findall(text):
        try:
            out.add(float(tok.replace('$', '').replace(',', '').rstrip('%')))
        except ValueError:
            pass
    return out


def check(answer_text, retrieved_ids, by_id, question=""):
    body   = CITE.sub(' ', answer_text)          # strip markers: "[method-06]" would read as 6
    source = "\n".join(by_id[i]["text"] for i in retrieved_ids)

    cited       = set(CITE.findall(answer_text))
    abstained   = bool(ABSTAIN.search(body))
    bad_cites   = cited - set(retrieved_ids)
    # Numbers echoed from the question aren't the model's — exclude them from both checks.
    unsupported = _numbers(body) - _numbers(source) - _numbers(question)
    # A refusal should contain no source content. Numbers echoed from the question don't count.
    leaked      = sorted(_numbers(body) - _numbers(question)) if abstained else []

    return {"cited": sorted(cited),
            "fabricated_citations": sorted(bad_cites),
            "unsupported_numbers":  sorted(unsupported),
            "leaked_in_refusal":    leaked,
            "uncited":              not cited and not abstained,
            "abstained":            abstained,
            "pass": not bad_cites and not unsupported and not leaked
                    and (bool(cited) or abstained)}


by_id = {c["id"]: c for c in chunks}


def render(text):
    """Replace [method-06] with a readable section name."""
    def sub(m):
        c = by_id.get(m.group(1))
        return f"[{c['heading'].split('>')[-1].strip()}]" if c else m.group(0)
    return CITE.sub(sub, text)


def ask(question, k=3, client="roselle", show_ids=False):
    text, retrieved = answer(question, k, client)
    report = check(text, [cid for _, cid in retrieved], by_id, question)

    print(f"RETRIEVED : {retrieved}")
    print(f"GUARDRAIL : {'PASS' if report['pass'] else 'FAIL'}")
    for key in ("fabricated_citations", "unsupported_numbers", "leaked_in_refusal", "uncited"):
        if report[key]:
            print(f"    {key}: {report[key]}")
    print()
    print(text if show_ids else render(text))
    return None                                  # nothing to echo in the cell output

## 7. Test cases

Three behaviours worth confirming, in order of importance.

**Grounded answer** — a question the corpus covers should produce a cited answer that passes.

**Correct abstention** — a question the corpus does not cover should be refused. Patient
counts live in the productivity CSV, not in either document, so this is the honest test of
whether rule 2 holds.

**Client isolation** — running a Roselle question with `client=None` restricts retrieval to
the shared method corpus. The engagement figures should become unreachable, and the answer
should shift to the general rule or refuse outright. This is the confidentiality guarantee,
and it is worth demonstrating rather than assuming.

In [ ]:
## Cell 8 — behavioural tests

print("=" * 72, "\n1. GROUNDED ANSWER\n")
ask("Why is the automobile add-back only 25 percent?")

print("\n" + "=" * 72, "\n2. CORRECT ABSTENTION (not in either corpus)\n")
ask("Did Harvey Seybold see more patients in 2023 than all other dentists?")

print("\n" + "=" * 72, "\n3. CLIENT ISOLATION (method corpus only)\n")
ask("What was the FY2022 owner excess compensation?", client=None)

---

## Next: integration

Two functions and two files port into the main pipeline: `embed()` and `search()`, plus
`chunks.json` and the corpus markdown. The tokenizer and model load once at startup, like the
CSVs do now.

**One insertion point.** `generate_llm_prompt()` gains a step — retrieve the top chunks and
inject them as a SOURCES block alongside the financial context it already builds. Everything
else is untouched: the CSV loaders, `detect_provider_in_question()`, the SDE metric
calculations. Numeric questions keep using the deterministic path, because retrieval would
make those worse. Retrieval only adds the *rules* the model previously could not see.

**One change `check()` will need.** It currently compares numbers in the answer against
numbers in the retrieved chunks. Once the prompt also carries CSV figures, the model will
cite those correctly and every one will be flagged as unsupported. Pass the full context to
the checker, not just the chunks:

```python
check(text, ids, by_id, question, extra_context=get_comprehensive_context())
```

Otherwise the first integrated run produces a wall of false failures.